In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Riyaz\\OneDrive\\Desktop\\text_summarizer\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Riyaz\\OneDrive\\Desktop\\text_summarizer'

In [5]:
## this is entity class
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file:Path
    unzip_dir:Path

In [6]:
from textSummarizer.constants import *

In [7]:
from textSummarizer.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath= PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params= read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

        def get_data_ingestion_config(self) -> DataIngestionConfig:
            config =self.config.data_ingestion

            create_directories([config.root_dir])

            data_ingestion_config = DataIngestionConfig(
                root_dir= config.root_dir,
                source_url=config.source_url,
                local_data_file=config.local_data_file,
                unzip_dir=config.unzip_dir
            )

            return data_ingestion_config

In [9]:
import os
import urllib.request as request
import zipfile
from textSummarizer.logging import logger
from textSummarizer.utils.common import get_size

In [10]:
class DataIngestion:
    def __init__(self,config : DataIngestionConfig):
        self.config =config


    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename,headers = request.urlretrieve(
                url=self.config.source_url,
                filename=self.config.local_data_file
            )

            logger.info(f"{filename} downaload with followig info : \n{headers}")
        else:
            logger.info(f"file laready exists of size : {get_size(Path(self.config.local_data_file))}")




    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracxs the zip file into the data directory 
        function returns none
        """

        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path,exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file,'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [13]:
try :
    config= ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion =DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise e 

[2026-05-11 09:54:01,453: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-11 09:54:01,456: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-11 09:54:01,458: INFO: common: created directory at: artifacts]
[2026-05-11 09:54:01,461: INFO: common: created directory at: artifacts/data_ingestion]
[2026-05-11 09:54:05,064: INFO: 3969783541: artifacts/data_ingestion/data.zip downaload with followig info : 
Connection: close
Content-Length: 7903594
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "dbc016a060da18070593b83afff580c9b300f0b6ea4147a7988433e04df246ca"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 5B00:66E1:2C1271:824793:6A0159DE
Accept-Ranges: bytes
Date: Mon, 11 May 2026 04:23:59 GMT
Via: 1.1 varnish
X-Served-By: cache-maa10226-MAA
X-Cache: MIS